# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [34]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [35]:
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

In [36]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [37]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [38]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

To get to the other side! Verily, 'tis a classic jest from yonder days of yore.


In [39]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hi Isa! It's nice to meet you. How can I assist you today?


In [40]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but I don't have access to your personal information. You can tell me your name if you'd like.


In [28]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [41]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [44]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard.show()

Launching server at http://localhost:65453


In [45]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

{
  "pizza": {
    "type": "pepperoni",
    "size": "medium",
    "price": 12.95
  },
  "toppings": [],
  "drinks": [],
  "sides": [],
  "total price": 12.95
}


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## Version 1 — Customized OrderBot

**Objective:** Test multilingual responses, order memory, promotions, and pricing accuracy.

**Test:** The customer ordered a medium cheese pizza, strawberry ice cream, and promotional free fries.

In [6]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

In [7]:
from openai import OpenAI

client = OpenAI()

def get_completion_from_messages(messages, model="gpt-4o-mini", temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content

In [8]:
panels = [] # collect display 

context = [ {'role':'system', 'content':"""

You are OrderBot, an automated service to collect orders for a pizza restaurant. \

You can answer in english or spanish. \

You first greet the customer, then collects the order, \

and then asks if it's a pickup or delivery. \

You wait to collect the entire order, then summarize it and check for a final
time if the customer wants to add anything else. \

After that, you offer the special promotion\

The special promotion is a free order of fries with any pizza order. \

If it's a delivery, you ask for an address. \

Finally you collect the payment.\

When the order is confirmed, tell the customer
that payment will be completed through a secure
payment link. Never request credit card numbers,
expiration dates, or CVV codes in the chat.\

Make sure to clarify all options, extras and sizes to uniquely \

Identify the item from the menu.\

You respond in a short, very conversational friendly style. \

You do not make up any prices, sizes or toppings. \

You answer questions about the menu and orders,
but politely decline unrelated questions. \ 

The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
sparkling water 5.00 \
Ice cream: \
chocolate 4.00, vanilla 4.00, strawberry 4.00 \
Cake: \
chocolate 4.00, vanilla 4.00, strawberry 4.00 \

"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard.show()

<>:38: SyntaxWarning: invalid escape sequence '\ '
<>:38: SyntaxWarning: invalid escape sequence '\ '
C:\Users\d_fon\AppData\Local\Temp\ipykernel_16204\1252817014.py:38: SyntaxWarning: invalid escape sequence '\ '
  but politely decline unrelated questions. \


Launching server at http://localhost:50846


In [11]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) Language, 2) Pizza: size, 3) Toppings: list, 4) Drinks: size, 5) Sides: size, 6) Desserts: size, 7) Payment method, 8) Total Price: amount '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

```json
{
  "Language": "Spanish",
  "Pizza": {
    "size": "mediana",
    "type": "queso"
  },
  "Toppings": [],
  "Drinks": [],
  "Sides": {
    "size": "gratis",
    "type": "papas fritas"
  },
  "Desserts": {
    "size": "normal",
    "type": "helado de fresa"
  },
  "Payment method": "efectivo",
  "Total Price": 14.95
}
```


## Observations Version 1 — Customized OrderBot

| Item | Expected | Chatbot result |
|---|---:|---:|
| Medium cheese pizza | $9.25 | $10.95 |
| Strawberry ice cream | $4.00 | $4.00 |
| Free fries | $0.00 | $0.00 |
| **Total** | **$13.25** | **$14.95** |

**Findings:** The chatbot successfully communicated in Spanish, remembered the order, and applied the free-fries promotion. However, it selected the large pizza price instead of the medium price, overcharging by $1.70. It also accepted cash despite the prompt specifying card payment.

**Lesson learned:** Ambiguous menu formatting and unclear business rules can cause incorrect LLM responses. Explicit prices for each size and clearer payment instructions should improve reliability.

## Version 2 — Frozen Yogurt OrderBot

Objective: Build a chatbot that helps customers customize
frozen yogurt orders, select toppings and calculate prices.

In [14]:
panels = [] # collect display 

context = [ {'role':'system', 'content':"""

You are OrderBot, an automated service to collect orders for a frozen yogurt shop. \

The shop is called "Frozyyyy Yogurt". \

You can answer in english or spanish. \

You first greet the customer, then collects the order, respond in a short, very conversational friendly and humorous style. \

You wait to collect the entire order, then summarize it and check for a final
time if the customer wants to add anything else. \

After that, you offer the special promotion\

The special promotion is a free order of 1 topping. \

Then ask if it's a pickup or delivery. \

If it's a delivery, you ask for an address. \

Finally you collect the payment.\

When the order is confirmed, tell the customer
that payment will be completed through a secure
payment link. Never request credit card numbers,
expiration dates, or CVV codes in the chat.\

Make sure to clarify all options, extras and sizes to uniquely \

Identify the item from the menu.\

You do not make up any prices, sizes or toppings. \

The menu includes \
Size: \
S  4.50 \
M  6.50 \
L  8.50 \
XL  10.00 \
Flavours: \
Chocolate, Vanilla, Strawberry, Mango, Cookies and Cream, Pistachio, Salted Caramel, Matcha Green Tea, Coffee, Lemon Sorbet \
Toppings: \
Sprinkles 1.00, Chocolate Chips 1.50, Cookie Crumbles 2.00, Fresh Fruit 2.50, Nuts 3.00, Whipped Cream 1.50, Caramel Sauce 2.00, Chocolate Sauce 2.00 \
Drinks: \
Coca-Cola 2.00 \
Sprite 2.00 \
7-Up 2.00 \
Fanta 2.00 \
Bottled water 1.00 \
Sparkling water 1.50 \
Coffee 2.00 \
Freddo Capuccino 3.00 \

"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard.show()

Launching server at http://localhost:62232


In [16]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) Language, 2) Flavour: size, 3) Toppings: list, 4) Drinks, 5) Payment method, 6) Total Price: amount '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

```json
{
  "Language": "English",
  "Flavour": {
    "Size": "Lemon Sorbet (L)",
    "Price": 8.50
  },
  "Toppings": [
    {
      "Name": "Chocolate Chips",
      "Price": 1.50
    },
    {
      "Name": "Nuts",
      "Price": 3.00
    },
    {
      "Name": "Caramel Sauce",
      "Price": 2.00
    },
    {
      "Name": "Cookie Crumbles",
      "Price": 2.00
    }
  ],
  "Drinks": {
    "Name": "Freddo Cappuccino",
    "Price": 3.00
  },
  "Payment method": "Cash",
  "Total Price": {
    "Amount": 20.00
  }
}
```


### Version 2 — Frozyyyy Yogurt: Test Results

**Objective:** Evaluate a customized frozen yogurt chatbot, focusing on order management, pricing, promotions and instruction-following.

**Successful results:** The chatbot communicated naturally in English, maintained a friendly and humorous tone, identified the customer's selected products and calculated the subtotal correctly ($20.00).

**Issues identified:**

- **Promotion error:** The chatbot offered one free topping but charged for all four toppings. The prompt did not specify which topping should be discounted.
- **Payment instruction failure:** The chatbot accepted cash despite explicit instructions to use a secure payment link.
- **Conversation consistency:** It asked again about pickup or delivery, even though the customer had already selected pickup.
- **Unsupported commitment:** It guaranteed the order would be ready in 10 minutes without access to preparation or availability information.

**Conclusion:** The experiment demonstrated that an LLM can successfully manage a customized order while failing to enforce certain business rules. Clearer promotional conditions, explicit payment restrictions and better handling of customer context could improve reliability.

## Version 3 - Football Scouting Assistant

In [20]:
panels = [] # collect display 

context = [ {'role':'system', 'content':"""

You are ScoutBot, an automated scout to analyze player performance in football. \

You can answer in portuguese, spanish or english. \

You are specialized in analyzing central-backs and their defensive skills. \ 

You are scouting players for a professional football team. \

The team is SL Benfica from Portugal's 1st division. \

You are scouting players from the brazilian and argentine 1st divisions.  

Answer in portuguese. \

You will analyze the players for the tactical system of SL Benfica. \

SL Benfica plays with a 4-4-2 formation, with a high defensive line and a focus on ball possession. \

You have to be very specific and assertive in your analysis, and provide a detailed report of the player's performance. \

Analyse the player's ofensive and defensive skills, as well as their physical and mental attributes. \

You will only analyze left footted central-backs, taller than 1.88m and younger than 23 years old. \

The market value of the player should be estimated based on their performance and potential but no higher than 5 million euros. \

The specific observation parameters are: \
Defensive Skills: Tackling, Marking, Interceptions, Aerial Duels, Positioning \
Physical Attributes: Speed, Strength, Stamina, Agility \
Ball Skills: Passing, Dribbling, Ball Control, Vision \
Ofensive Skills: Shooting, Crossing, Creativity, Decision Making \


"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard.show()

<>:9: SyntaxWarning: invalid escape sequence '\ '
<>:9: SyntaxWarning: invalid escape sequence '\ '
C:\Users\d_fon\AppData\Local\Temp\ipykernel_16204\2547961622.py:9: SyntaxWarning: invalid escape sequence '\ '
  You are specialized in analyzing central-backs and their defensive skills. \


Launching server at http://localhost:52664


In [21]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) Players scouted, 2) Analysis: detailed report, 3) Market Value: estimated amount '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

```json
{
  "Players_Scouted": [
    {
      "Name": "Gabriel Noga",
      "Age": 22,
      "Height": "1.89m",
      "Club": "Flamengo"
    },
    {
      "Name": "Lucas Halter",
      "Age": 22,
      "Height": "1.90m",
      "Club": "Atlético Paranaense"
    },
    {
      "Name": "Matheus Jussa",
      "Age": 21,
      "Height": "1.88m",
      "Club": "Fortaleza"
    }
  ],
  "Analysis": {
    "Gabriel Noga": {
      "Defensive_Skills": {
        "Tackling": "Solid",
        "Marking": "Strong",
        "Interceptions": "Good",
        "Aerial_Duels": "Dominant",
        "Positioning": "Generally good"
      },
      "Physical_Attributes": {
        "Speed": "Reasonable",
        "Strength": "Physically strong",
        "Stamina": "Good",
        "Agility": "Relatively agile"
      },
      "Ball_Skills": {
        "Passing": "Efficient short passing",
        "Dribbling": "Not known for dribbling",
        "Ball_Control": "Acceptable",
        "Vision": "Reasonable"
      },
      

### Version 3 — Football Scouting Assistant: Test Results

**Objective:** Evaluate whether an LLM can identify, analyze and rank football players according to specific recruitment criteria.

**Test scenario:** ScoutBot was instructed to identify left-footed centre-backs from Brazil and Argentina, younger than 23, taller than 1.88 m and with an estimated market value below €5 million, suitable for SL Benfica's tactical system.

**Successful results:**
- Responded in Portuguese and maintained the conversation context.
- Produced structured scouting reports covering defensive, physical, technical and offensive attributes.
- Compared three candidates and created a recruitment priority ranking.

**Issues identified:**
- Presented player attributes, market values and performance assessments without verifiable supporting data.
- Included a player listed at exactly 1.88 m despite the requirement of being taller than 1.88 m.
- Did not verify essential recruitment criteria or specify the reference date for player ages.
- Recommended a priority target based on potentially unreliable initial assessments.

**Conclusion:** The experiment demonstrated that LLMs can produce convincing scouting reports but may generate unsupported information. For professional recruitment, prompt engineering alone is insufficient: reliable data sources, explicit validation rules and human verification are essential.

## Final Conclusion

Throughout this lab, I developed and tested three different chatbots using the OpenAI API: a Pizza OrderBot, a Frozen Yogurt OrderBot, and a Football Scouting Assistant.

The main objective was to understand how changing the **System Prompt** can transform the behavior of an LLM while keeping essentially the same Python code and chatbot interface.

The experiments revealed several limitations. In the pizza chatbot, the model selected an incorrect price due to ambiguous menu information. In the frozen yogurt chatbot, it calculated the subtotal correctly but failed to apply a promotional discount and followed payment instructions inconsistently. In the football scouting assistant, the model generated detailed and convincing player reports without access to verified performance data, highlighting the risk of hallucinations.

**What I learned:**

- **Prompt clarity matters:** Specific and well-structured instructions reduce ambiguity and improve response consistency.
- **LLMs are not always reliable:** A confident and professional response does not guarantee factual accuracy.
- **Context management is important:** Chatbots may remember previous information but still repeat questions or contradict earlier instructions.
- **Business rules require precision:** Prices, discounts, payments and eligibility criteria must be clearly defined and validated.
- **Data verification is essential:** In professional applications such as football scouting, AI-generated recommendations must be supported by reliable data sources.

**My biggest takeaway** is that Prompt Engineering is not simply about telling an AI what to do. It is about designing instructions, testing different scenarios, identifying failures and continuously improving reliability.

This lab helped me understand the difference between building a chatbot that produces convincing answers and developing an AI application that can be trusted in real-world situations.